<a href="https://colab.research.google.com/github/tal21-linares/curso-ia-para-economia/blob/main/5_Taller_Ensamble.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<a href="https://colab.research.google.com/github/LinaMariaCastro/curso-ia-para-economia/blob/main/clases/5_Aprendizaje_supervisado/5_Taller_Ensamble.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Inteligencia Artificial con Aplicaciones en Economía I**

- 👩‍🏫 **Profesora:** [Lina María Castro](https://www.linkedin.com/in/lina-maria-castro)  
- 📧 **Email:** [lmcastroco@gmail.com](mailto:lmcastroco@gmail.com)  
- 🎓 **Universidad:** Universidad Externado de Colombia - Facultad de Economía

# **Taller Ensamble: Random Forest y Gradient Boosting**

**IMPORTANTE**: Guarda una copia de este notebook en tu Google Drive o computador.

**Taller en grupos de 3**

**Nombres estudiantes:**

- Talia Linares
- David Velandia

**Forma de entrega:**

- Nombrar el archivo de la siguiente forma: “Taller_Ensamble_apellidos.ipynb”.
- Suba el Jupyter Notebook a su cuenta en Github y envíe el link en el siguiente Forms: https://forms.cloud.microsoft/r/Hm6L1UMD03.

**IMPORTANTE:** No se recibirán talleres en Google Colab, el notebook debe estar subido en Github.

**Plazo de entrega:**

12 de mayo de 2026, máximo a las 11:59 p.m. Tenga en cuenta que luego de esa hora el formulario en forms se cierra. El Jupupyter Notebook también debe quedar subido en Github antes de esa hora.

**Instrucciones Generales:**

Completa el código en las celdas marcadas con `### TU CÓDIGO AQUÍ ###`. Puedes añadir más celdas si lo requieres.

## Caso de Consultoría: Ames Real Estate Solutions

**Contexto:** Eres un consultor de datos contratado por **Ames Real Estate Solutions**. Tu misión es construir un *pipeline* de Machine Learning robusto para predecir el precio de venta (`SalePrice`) de las viviendas en la ciudad de Ames, Iowa.

**Objetivos:**
1.  Construir un *pipeline* de preprocesamiento profesional usando `ColumnTransformer` y `OneHotEncoder`.
2.  Entrenar y comparar el desempeño en prueba de 3 modelos de regresión: árbol de decisión, random forest y gradient boosting.
3.  Evaluar los modelos usando **R-cuadrado (R²)** y **RMSE** (Raíz del Error Cuadrático Medio).
4.  Optimizar uno de los modelos, el de Random Forest, usando `GridSearchCV` para mejorar su rendimiento en prueba.

### Paso 1: Configuración Inicial

In [3]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

from sklearn.metrics import mean_squared_error, r2_score

Mejorar visualización de dataframes y gráficos

In [4]:
# Que muestre todas las columnas
pd.options.display.max_columns = None
# En los dataframes, mostrar los float con dos decimales
pd.options.display.float_format = '{:,.2f}'.format

# Configuraciones para una mejor visualización
sns.set(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

In [5]:
# Cargar el dataset
url = 'http://jse.amstat.org/v19n3/decock/AmesHousing.txt'
df = pd.read_csv(url, sep='\t')

print(f"Dataset cargado con {df.shape[0]} filas y {df.shape[1]} columnas.")
df.head()

Dataset cargado con 2930 filas y 82 columnas.


,Order,PID,MS SubClass,MS Zoning,Lot Frontage,Lot Area,Street,Alley,Lot Shape,Land Contour,Utilities,Lot Config,Land Slope,Neighborhood,Condition 1,Condition 2,Bldg Type,House Style,Overall Qual,Overall Cond,Year Built,Year Remod/Add,Roof Style,Roof Matl,Exterior 1st,Exterior 2nd,Mas Vnr Type,Mas Vnr Area,Exter Qual,Exter Cond,Foundation,Bsmt Qual,Bsmt Cond,Bsmt Exposure,BsmtFin Type 1,BsmtFin SF 1,BsmtFin Type 2,BsmtFin SF 2,Bsmt Unf SF,Total Bsmt SF,Heating,Heating QC,Central Air,Electrical,1st Flr SF,2nd Flr SF,Low Qual Fin SF,Gr Liv Area,Bsmt Full Bath,Bsmt Half Bath,Full Bath,Half Bath,Bedroom AbvGr,Kitchen AbvGr,Kitchen Qual,TotRms AbvGrd,Functional,Fireplaces,Fireplace Qu,Garage Type,Garage Yr Blt,Garage Finish,Garage Cars,Garage Area,Garage Qual,Garage Cond,Paved Drive,Wood Deck SF,Open Porch SF,Enclosed Porch,3Ssn Porch,Screen Porch,Pool Area,Pool QC,Fence,Misc Feature,Misc Val,Mo Sold,Yr Sold,Sale Type,Sale Condition,SalePrice
0,1,526301100,20,RL,141.00,31770,Pave,NaN,IR1,Lvl,AllPub,Corner,Gtl,NAmes,Norm,Norm,1Fam,1Story,6,5,1960,1960,Hip,CompShg,BrkFace,Plywood,Stone,112.00,TA,TA,CBlock,TA,Gd,Gd,BLQ,639.00,Unf,0.00,441.00,"1,080.00",GasA,Fa,Y,SBrkr,1656,0,0,1656,1.00,0.00,1,0,3,1,TA,7,Typ,2,Gd,Attchd,"1,960.00",Fin,2.00,528.00,TA,TA,P,210,62,0,0,0,0,NaN,NaN,NaN,0,5,2010,WD,Normal,215000
1,2,526350040,20,RH,80.00,11622,Pave,NaN,Reg,Lvl,AllPub,Inside,Gtl,NAmes,Feedr,Norm,1Fam,1Story,5,6,1961,1961,Gable,CompShg,VinylSd,VinylSd,NaN,0.00,TA,TA,CBlock,TA,TA,No,Rec,468.00,LwQ,144.00,270.00,882.00,GasA,TA,Y,SBrkr,896,0,0,896,0.00,0.00,1,0,2,1,TA,5,Typ,0,NaN,Attchd,"1,961.00",Unf,1.00,730.00,TA,TA,Y,140,0,0,0,120,0,NaN,MnPrv,NaN,0,6,2010,WD,Normal,105000
2,3,526351010,20,RL,81.00,14267,Pave,NaN,IR1,Lvl,AllPub,Corner,Gtl,NAmes,Norm,Norm,1Fam,1Story,6,6,1958,1958,Hip,CompShg,Wd Sdng,Wd Sdng,BrkFace,108.00,TA,TA,CBlock,TA,TA,No,ALQ,923.00,Unf,0.00,406.00,"1,329.00",GasA,TA,Y,SBrkr,1329,0,0,1329,0.00,0.00,1,1,3,1,Gd,6,Typ,0,NaN,Attchd,"1,958.00",Unf,1.00,312.00,TA,TA,Y,393,36,0,0,0,0,NaN,NaN,Gar2,12500,6,2010,WD,Normal,172000
3,4,526353030,20,RL,93.00,11160,Pave,NaN,Reg,Lvl,AllPub,Corner,Gtl,NAmes,Norm,Norm,1Fam,1Story,7,5,1968,1968,Hip,CompShg,BrkFace,BrkFace,NaN,0.00,Gd,TA,CBlock,TA,TA,No,ALQ,"1,065.00",Unf,0.00,"1,045.00","2,110.00",GasA,Ex,Y,SBrkr,2110,0,0,2110,1.00,0.00,2,1,3,1,Ex,8,Typ,2,TA,Attchd,"1,968.00",Fin,2.00,522.00,TA,TA,Y,0,0,0,0,0,0,NaN,NaN,NaN,0,4,2010,WD,Normal,244000
4,5,527105010,60,RL,74.00,13830,Pave,NaN,IR1,Lvl,AllPub,Inside,Gtl,Gilbert,Norm,Norm,1Fam,2Story,5,5,1997,1998,Gable,CompShg,VinylSd,VinylSd,NaN,0.00,TA,TA,PConc,Gd,TA,No,GLQ,791.00,Unf,0.00,137.00,928.00,GasA,Gd,Y,SBrkr,928,701,0,1629,0.00,0.00,2,1,3,1,TA,6,Typ,1,TA,Attchd,"1,997.00",Fin,2.00,482.00,TA,TA,Y,212,34,0,0,0,0,NaN,MnPrv,NaN,0,3,2010,WD,Normal,189900


Para este taller, nos enfocaremos en 6 variables predictoras + 1 variable objetivo.

- **Variable Objetivo (Target):**

  - **SalePrice (Precio de Venta):** Es una variable numérica continua que representa el precio final de la transacción de la vivienda en dólares.

- **Variables Predictoras (Features):**

Hemos seleccionado un conjunto mixto de 3 variables numéricas y 3 variables categóricas que intuitivamente tienen un fuerte impacto en el precio:

*Numéricas:*

  - **Overall Qual (Calidad General):** Es una calificación en una escala de 1 a 10 que resume la calidad general del material y los acabados de la casa. Es una medida directa de "lujo" y "calidad de construcción". Una casa con acabados de alta gama (calificación 9 o 10) valdrá mucho más que una con acabados básicos (calificación 4 o 5), incluso si tienen el mismo tamaño.

  - **Gr Liv Area (Área Habitable):** Es el área total en pies cuadrados de los espacios habitables que están sobre el nivel del suelo. Esta variable captura el "tamaño" útil de la vivienda. En igualdad de condiciones, una casa más grande es más valiosa.

  - **Year Built (Año de Construcción):** El año en que la casa fue construida originalmente. Representa la antigüedad de la casa. Las casas más nuevas suelen tener diseños más modernos, mejor eficiencia energética y menos necesidad de reparaciones inmediatas, lo que generalmente aumenta su valor.

*Categóricas:*

  - **Neighborhood (Barrio):** La ubicación física (el barrio) dentro de la ciudad de Ames. Esta variable captura miles de factores ocultos (efectos fijos) como la calidad de las escuelas, la seguridad, el prestigio social, el acceso a parques y el tiempo de desplazamiento.

  - **House Style (Estilo de Casa):** El estilo de la vivienda (ej. 1Story - 1 piso, 2Story - 2 pisos, SLvl - Niveles divididos). Diferentes estilos de construcción tienen diferentes costos y atraen a diferentes segmentos de compradores.

  - **Exter Qual (Calidad Exterior):** Calificación de la calidad de los materiales en el exterior de la casa (ej. Ex - Excelente, Gd - Bueno, TA - Típico/Promedio, Fa - Justo). Mide la calidad del "cascarón" de la casa y el "atractivo visual".

In [6]:
features = [
    'Overall Qual',  # Numérica (ordinal)
    'Gr Liv Area',   # Numérica
    'Year Built',    # Numérica
    'Neighborhood',  # Categórica
    'House Style',   # Categórica
    'Exter Qual'     # Categórica (ordinal)
]
target = 'SalePrice'
df_model = df[features + [target]].copy()
df_model.head()

,Overall Qual,Gr Liv Area,Year Built,Neighborhood,House Style,Exter Qual,SalePrice
0,6,1656,1960,NAmes,1Story,TA,215000
1,5,896,1961,NAmes,1Story,TA,105000
2,6,1329,1958,NAmes,1Story,TA,172000
3,7,2110,1968,NAmes,1Story,Gd,244000
4,5,1629,1997,Gilbert,2Story,TA,189900


### Paso 2: Preprocesamiento y Creación del Pipeline

Pasos:

1.  Separar `X` e `y`.
2.  Separar en `train` y `test` **antes** de cualquier transformación para evitar fuga de datos.
3.  Definir un `ColumnTransformer` que sepa qué hacer con las columnas numéricas y categóricas.
4.  Ajustar (`fit_transform`) el transformador SÓLO en los datos de `train`.

In [7]:
X = df[features]
y = df['SalePrice']

In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print(f"Tamaño de X_train: {X_train.shape}")
print(f"Tamaño de X_test: {X_test.shape}")

Tamaño de X_train: (2344, 6)
Tamaño de X_test: (586, 6)


In [9]:
numerical_features = X.select_dtypes(include=['int64', 'float64']).columns
categorical_features = X.select_dtypes(include=['object']).columns

print(f"Columnas Numéricas: {list(numerical_features)}")
print(f"Columnas Categóricas: {list(categorical_features)}")

Columnas Numéricas: ['Overall Qual', 'Gr Liv Area', 'Year Built']
Columnas Categóricas: ['Neighborhood', 'House Style', 'Exter Qual']


In [10]:
categorical_transformer = OneHotEncoder(handle_unknown='ignore')

preprocessor = ColumnTransformer(
    transformers=[
        ('cat', categorical_transformer, categorical_features)
    ],
    remainder='passthrough'
)

In [11]:
X_train_processed = preprocessor.fit_transform(X_train)

X_test_processed = preprocessor.transform(X_test)

print(f"\nForma de X_train procesado: {X_train_processed.shape}")
print(f"Forma de X_test procesado: {X_test_processed.shape}")


Forma de X_train procesado: (2344, 43)
Forma de X_test procesado: (586, 43)


### Paso 3: Modelos Base (Baseline)

Entrenemos 3 modelos: árbol de decisión, random forest y gradient boosting.

In [12]:
modelo_dt = DecisionTreeRegressor(random_state=42)

modelo_dt.fit(X_train_processed, y_train)

y_pred_dt = modelo_dt.predict(X_test_processed)

rmse_dt = np.sqrt(mean_squared_error(y_test, y_pred_dt))
r2_dt = r2_score(y_test, y_pred_dt)

print(f"RMSE Decision Tree: {rmse_dt:.2f}")
print(f"R2 Decision Tree: {r2_dt:.4f}")

RMSE Decision Tree: 35544.40
R2 Decision Tree: 0.8424


In [13]:
modelo_rf = RandomForestRegressor(random_state=42)

modelo_rf.fit(X_train_processed, y_train)

y_pred_rf = modelo_rf.predict(X_test_processed)

rmse_rf = np.sqrt(mean_squared_error(y_test, y_pred_rf))
r2_rf = r2_score(y_test, y_pred_rf)

print(f"RMSE Random Forest: {rmse_rf:.2f}")
print(f"R2 Random Forest: {r2_rf:.4f}")

RMSE Random Forest: 28464.75
R2 Random Forest: 0.8989


In [14]:
modelo_gb = GradientBoostingRegressor(random_state=42)

modelo_gb.fit(X_train_processed, y_train)

y_pred_gb = modelo_gb.predict(X_test_processed)

rmse_gb = np.sqrt(mean_squared_error(y_test, y_pred_gb))
r2_gb = r2_score(y_test, y_pred_gb)

print(f"RMSE Gradient Boosting: {rmse_gb:.2f}")
print(f"R2 Gradient Boosting: {r2_gb:.4f}")

RMSE Gradient Boosting: 32383.26
R2 Gradient Boosting: 0.8692


### Paso 4: Comparación de Métricas (Baseline)

In [15]:
resultados = pd.DataFrame({
    'Modelo': ['Decision Tree', 'Random Forest', 'Gradient Boosting'],
    'RMSE': [rmse_dt, rmse_rf, rmse_gb],
    'R2': [r2_dt, r2_rf, r2_gb]
})

resultados = resultados.sort_values(by='RMSE')

print(resultados)

              Modelo      RMSE   R2
1      Random Forest 28,464.75 0.90
2  Gradient Boosting 32,383.26 0.87
0      Decision Tree 35,544.40 0.84


¿Cuál modelo tuvo un mejor desempeño en el dataset de prueba (menor RMSE, mayor R²)?

El modelo con mejor desempeño fue Random Forest, ya que obtuvo el menor RMSE (28,464.75) y el mayor R² (0.90) en el conjunto de prueba. Esto significa que sus predicciones fueron las más cercanas a los precios reales de las viviendas y que logró explicar aproximadamente el 90% de la variabilidad del precio de venta. Por otro lado, aunque Gradient Boosting y Decision Tree también tuvieron resultados aceptables, presentaron errores más altos y una menor capacidad explicativa.

### Paso 5: Optimización con GridSearchCV para Random Forest

La búsqueda de los hiperparámetros óptimos con `GridSearchCV` se puede realizar para los 3 modelos, sin embargo, como esto toma tiempo, en este taller lo realizaremos solo para el modelo Random Forest.

In [16]:
param_grid_rf = {
    'n_estimators': [100, 200],
    'max_depth': [10, 20, None],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2]
}

rf = RandomForestRegressor(random_state=42)

grid_search_rf = GridSearchCV(
    estimator=rf,
    param_grid=param_grid_rf,
    cv=3,
    scoring='neg_mean_squared_error',
    n_jobs=-1,
    verbose=1
)

grid_search_rf.fit(X_train_processed, y_train)

print("¡Búsqueda completada!")
print(f"Mejores parámetros encontrados: {grid_search_rf.best_params_}")

Fitting 3 folds for each of 24 candidates, totalling 72 fits
¡Búsqueda completada!
Mejores parámetros encontrados: {'max_depth': 10, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 200}


Investiga por qué en GridSearchCV se usa como métrica el error cuadrático medio **negativo**. A continuación escribe la explicación.

En GridSearchCV se utiliza la métrica negative mean squared error porque Scikit-Learn está diseñado para maximizar métricas de evaluación, es decir, considera que valores más altos representan mejores modelos. Sin embargo, el Mean Squared Error (MSE) funciona al contrario: entre menor sea el error, mejor es el modelo. Para solucionar esto, Scikit-Learn multiplica el MSE por -1 y así convierte el problema en una maximización. De esta forma, el modelo con el valor “menos negativo” será realmente el que tenga el menor error cuadrático medio y, por tanto, el mejor desempeño.

### Paso 6: Evaluación Final del Modelo Random Forest Optimizado

Ahora, usemos nuestro modelo ganador y optimizado (best_estimator_) para hacer predicciones en el set de prueba y ver si el RMSE mejoró respecto a nuestro Random Forest base.

In [17]:
best_rf_model = grid_search_rf.best_estimator_

y_pred_rf_optimized = best_rf_model.predict(X_test_processed)

rmse_rf_optimized = np.sqrt(mean_squared_error(y_test, y_pred_rf_optimized))
r2_rf_optimized = r2_score(y_test, y_pred_rf_optimized)

print(f"RMSE RF Optimizado: {rmse_rf_optimized:.2f}")
print(f"R2 RF Optimizado: {r2_rf_optimized:.4f}")

print(f"RMSE Random Forest Base: {rmse_rf:.2f}")
print(f"RMSE Random Forest Optimizado: {rmse_rf_optimized:.2f}")

reduccion_rmse = rmse_rf - rmse_rf_optimized

print(f"Reducción del RMSE: {reduccion_rmse:.2f}")

RMSE RF Optimizado: 29297.99
R2 RF Optimizado: 0.8929
RMSE Random Forest Base: 28464.75
RMSE Random Forest Optimizado: 29297.99
Reducción del RMSE: -833.25


### Paso 7: Conclusión de Consultoría

Basado en los resultados finales, responde a tu cliente.

**Pregunta 1:** ¿Cuánto logramos reducir el error (RMSE) al optimizar el modelo Random Forest con GridSearchCV? (Compara el RMSE base de RF vs. el RMSE final).


En este caso, la optimización del modelo Random Forest mediante GridSearchCV no logró mejorar el desempeño del modelo base. El Random Forest original obtuvo un RMSE de 28,464.75, mientras que el modelo optimizado presentó un RMSE de 29,297.99, por lo que el modelo base terminó siendo ligeramente más preciso en el conjunto de prueba. Aunque el GridSearchCV permitió probar diferentes combinaciones de hiperparámetros, los resultados muestran que la configuración inicial del modelo ya tenía un desempeño bastante sólido y generalizaba mejor para este dataset.


**Pregunta 2:** ¿Por qué fue crucial usar `ColumnTransformer` y hacer el `train_test_split` antes de transformar los datos? ¿Qué problema evitamos?

Fue crucial usar ColumnTransformer porque permitió transformar correctamente las variables categóricas mediante OneHotEncoding sin modificar las variables numéricas. Además, realizar primero el train_test_split evitó el problema de data leakage (fuga de datos). Si las transformaciones se hubieran aplicado antes de dividir el dataset, el modelo habría tenido acceso indirecto a información del conjunto de prueba durante el entrenamiento, generando métricas artificialmente optimistas y poco confiables. Separar los datos antes del preprocesamiento garantiza una evaluación más realista y correcta del desempeño del modelo.